In [ ]:
# import the library
from pynq import Overlay     # import the overlay
from pynq import allocate    # import for CMA (contingeous memory allocation)
from pynq import DefaultIP   # import the ip connector library for extension
from pynq import Interrupt
import asyncio
import numpy as np
import os
import dfx4ml.cap         as cap
import dfx4ml.mem_alloc   as dataAlloc  # import the memory allocation library
import dfx4ml.dfx_unified as dfx_unified
import time

In [ ]:
PRJ_DIR = os.getcwd()
PRJ_HW_DIR = PRJ_DIR + '/hw/'
PRJ_TC_DIR = PRJ_DIR + '/sw/'
DFX_CONFIG_FILE = 'dfx_ctrl_meta.txt'

FULL_BS_NAME = 'system.bin'
PAR_BS_NAME_0 = 'par0.bin'  ###### dma to magic stream 1
INPUT_DATA_NAME = "input_x.npy"

AMT_QUERY = 100
INPUT_SHAPE = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes
OUTPUT_SHAPE = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes
AMT_SLOT = 1

input_x = np.arange(AMT_QUERY, dtype=np.int32).reshape(INPUT_SHAPE)
np.save(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME), input_x)


In [2]:
cap.change_pl_config_mode("pcap", True, "")

CHANGE CMD STDOUT: 0xFFCA3008 0xFFFFFFFF 0x1

CHANGE CMD ERROR : 
--------------------------------
TRIGGER CMD STDOUT: 0xFFCA3008

TRIGGER CMD ERROR : 
--------------------------------
READ CMD STDOUT: 0x1

READ CMD ERROR : 
--------------------------------


In [3]:
#### load the overlay
overlay  = Overlay(PRJ_HW_DIR + FULL_BS_NAME)

In [4]:
#### create the interrupt pin
overlay.interrupt_pins


{'magicSeq/MagicSeqTopIntr_0/hw_intr': {'controller': 'axi_intc_0',
  'index': 0,
  'fullpath': 'magicSeq/MagicSeqTopIntr_0/hw_intr'},
 'axi_intc_0/intr': {'controller': 'axi_intc_0',
  'index': 0,
  'fullpath': 'axi_intc_0/intr'}}

In [5]:
my_interrupt = Interrupt('dfx_unified_0/dfx_intr')  # index 0 from your mapping

In [ ]:
dfx_unifed_ip = overlay.dfx_unified_0

In [6]:
#### get the device
dmaIp      = dfx_unifed_ip.dfx_dma
dfx_ctrl  = dfx_unifed_ip.dfx_ctrl
dfx_mng = dfx_unifed_ip.dfx_mng

In [7]:
#### configure the dfx controller ip to match the address space
dfx_ctrl.config(PRJ_HW_DIR + DFX_CONFIG_FILE)
print("regIdxSize = ", dfx_ctrl.BLS_REGID)

regbank detect index (7, 8)
regbank detect index (2, 6)
regIdxSize =  5


In [8]:
### change reconfigure mode
cap.change_pl_config_mode("icap", True, "")

CHANGE CMD STDOUT: 0xFFCA3008 0xFFFFFFFF 0x0

CHANGE CMD ERROR : 
--------------------------------
TRIGGER CMD STDOUT: 0xFFCA3008

TRIGGER CMD ERROR : 
--------------------------------
READ CMD STDOUT: 0x0

READ CMD ERROR : 
--------------------------------


In [9]:
dfx_ctrl.print_status()

[get status register] @ 0x0
>>status of the system vs0
-------> Is device shutdown:  1
-------> current error code:  0x0
-------> active RM_ID      :  0x0
-------> state      :  0x1


In [10]:
#### shutdown all system
dfx_mng .shutdown_engine()
dfx_ctrl.shutdown_engine()

[cmd] shutdown the engine
[cmd] shutdown successfully
shutdown dfx Controller
[set ctrl register] @ 0x0  with command  0x0


In [11]:
# get physical address of dma and dfx controller
dmaPhyAddr     =  0x0002_0000 #overlay.ip_dict['dataMovement/axi_dma_0']['phys_addr']
dfxCtrlPhyAddr =  0 #overlay.ip_dict['PRcontroller/dfx_controller_0']['phys_addr']

print("dma physical address: ", hex(dmaPhyAddr))
print("dfx  Ctrl physical address: ", hex(dfxCtrlPhyAddr))

dma physical address:  0xa0040000
dfx  Ctrl physical address:  0xa0020000


In [12]:
##### initialize magic seq
print("------ before init magic seq------")
print(dfx_mng.print_debug())

print("------ init magic sequence METADATA bank 0 -------------------------")
dfx_mng.set_end_cnt(AMT_SLOT - 1) ### use the last index
dfx_mng.set_dma_addr(dmaPhyAddr)
dfx_mng.set_dfx_addr(dfxCtrlPhyAddr)
dfx_mng.set_intr_ena(1)
dfx_mng.set_intr(1)  # woc  command 1 to set the interrupt to 0
dfx_mng.set_round_trip(0)  # set round trip to 0, no need to wait for the dma to finish
inputX = np.load(PRJ_TC_DIR + INPUT_DATA_NAME)
if(inputX.shape != INPUT_SHAPE):
    raise Exception(f"inputX shape is {inputX} expect {INPUT_SHAPE}")

#inputX = np.random.rand(*INPUT_SHAPE).astype(np.float32)
print("-------------init all data buffer -------------")
buf_input   , buf_input_phya   , buf_input_sz    = dataAlloc.alloc_data_uint(alloc_shape= INPUT_SHAPE , alloc_type=np.int32, input_x = inputX)
buf_out     , buf_out_phy      , buf_out_sz      = dataAlloc.alloc_data_uint(alloc_shape= OUTPUT_SHAPE, alloc_type=np.int32)
buf_input.flush()
print("------------- init dfx mng ------------------")
#                         [srcPhyAddr    ,        srcSz,  dstPhyAddr,      dstSz,st,pr,loadMask, storeMask, intrMask]
dfx_mng.set_whole_slot(0, [buf_input_phya, buf_input_sz, buf_out_phy, buf_out_sz, 0, 0,  0b0001, 0b0001, 0])

print("------------- after init magic seq------")
print(dfx_mng.print_debug())

------ before init magic seq------
----- MAIN STATUS ------------------
--------> STATUS =  SHUTDOWN
--------> MAINCNT =  0
--------> ENDCNT  =  0
--------> DMAADDR  =  0x0
--------> DFXADDR  =  0x0
--------> INTR_ENA =  0x0
--------> INTR     =  0x0
--------> ROUND_TRIP =  0x0
----- SLOT DATA ------------------
------> slot 0 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 1 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 2 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 3 :
      

In [13]:
##### initialize dfx controller
print("------ allocate bit steram CMA for each trigger ------")

######## set trigger 0
d0_ip_buf, d0_addr, d0_size = \
    dfx_ctrl.allocate_bit_stream_cma(PRJ_HW_DIR + PAR_BS_NAME_0)

------ allocate bit steram CMA for each trigger ------
>>allocateBitStream
opening file  /home/xilinx/jupyter_notebooks/tanawin/dfx4ml_magicSkipper/hw/skipper1.bin
copying the data
copy complete
file size  18681736
---------------------------------
>>allocateBitStream
opening file  /home/xilinx/jupyter_notebooks/tanawin/dfx4ml_magicSkipper/hw/skipper2.bin
copying the data
copy complete
file size  18681736
---------------------------------


In [14]:
##### initialize dfx controller2
dfx_ctrl.set_simple_meta_data(0, d0_addr, d0_size)

setting RM Mapping to  0
[set RM MAP] @ 0x80  info  0x0
setting RM INFO to  0
control value for active low reset is  0x10
[get RM INFO] bsIdxAddr@ 0x100  ctrlAddr@ 0x104
setting BS INFO to  0  with streamAddress:  2017460224  with size:  18681736
[get BS INFO] streamAddr@ 0x184  sizeAddr@ 0x188
setting RM Mapping to  1
[set RM MAP] @ 0x84  info  0x1
setting RM INFO to  1
control value for active low reset is  0x10
[get RM INFO] bsIdxAddr@ 0x108  ctrlAddr@ 0x10c
setting BS INFO to  1  with streamAddress:  2036334592  with size:  18681736
[get BS INFO] streamAddr@ 0x194  sizeAddr@ 0x198


In [15]:
##### check dfx controller3
dfx_ctrl.print_status()
dfx_ctrl.print_simple_meta_data(0)

[get status register] @ 0x0
>>status of the system vs0
-------> Is device shutdown:  1
-------> current error code:  0x0
-------> active RM_ID      :  0x0
-------> state      :  0x1
get metadata info for row 0
[get RM MAP] @ 0x80
RM MAPPER:  0
[get RM INFO] bsIdxAddr@ 0x100  ctrlAddr@ 0x104
RM INFO  :  (0, 16)
[get BS INFO] streamAddr@ 0x184  sizeAddr@ 0x188
BS INFO  :  (0, 2017460224, 18681736)
get metadata info for row 1
[get RM MAP] @ 0x84
RM MAPPER:  1
[get RM INFO] bsIdxAddr@ 0x108  ctrlAddr@ 0x10c
RM INFO  :  (1, 16)
[get BS INFO] streamAddr@ 0x194  sizeAddr@ 0x198
BS INFO  :  (0, 2036334592, 18681736)


In [16]:
dfx_ctrl.trig(0)
dfx_ctrl.restart_no_status()

trig the rmId  0
[set Ctrl Trigger] @ 0x4
restart the dfx Controller with no status
[set ctrl register] @ 0x0  with command  0x1


In [17]:
dfx_ctrl.print_status()

[get status register] @ 0x0
>>status of the system vs0
-------> Is device shutdown:  0
-------> current error code:  0x0
-------> active RM_ID      :  0x0
-------> state      :  0x7


In [18]:
##### start dfx controller3
async def startExecAndWait4Intr():
    start_time = time.perf_counter()  # Start timing
    dfx_mng.clear_intr()
    dfx_mng.start_engine()
    while True:
        await my_interrupt.wait()
        end_time = time.perf_counter()
        print("interrupt")
        print(f"Elapsed time: {end_time - start_time:.6f} seconds")
        break

In [19]:
loop2 = asyncio.get_event_loop()

In [20]:
task2 = loop2.create_task(startExecAndWait4Intr())
loop2.run_until_complete(task2)

[cmd] clear the interrupt
[cmd] clear the interrupt successfully
[cmd] start the engine
[cmd] start the successfully
interrupt
Elapsed time: 0.095705 seconds


In [21]:
print(dfx_mng.print_debug())

----- MAIN STATUS ------------------
--------> STATUS =  SHUTDOWN
--------> MAINCNT =  0
--------> ENDCNT  =  1
--------> DMAADDR  =  0xa0040000
--------> DFXADDR  =  0xa0020000
--------> INTR_ENA =  0x1
--------> INTR     =  0x1
--------> ROUND_TRIP =  0x23d6
----- SLOT DATA ------------------
------> slot 0 :
        srcAddr   : 0x77dd0000,  srcSize   : 0xffc0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x474444
        loadMask  : 0b1
        storeMask : 0b110
        stIntrMask: 0b110
------> slot 1 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x77de0000,  desSize   : 0xffc0
        status    : 0x0
        profileCnt: 0x474442
        loadMask  : 0b110
        storeMask : 0b1
        stIntrMask: 0b1
------> slot 2 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
-----

In [22]:
dfx_mng.shutdown_engine()

[cmd] shutdown the engine
[cmd] shutdown successfully


In [23]:
print(dfx_mng.print_debug())

----- MAIN STATUS ------------------
--------> STATUS =  SHUTDOWN
--------> MAINCNT =  0
--------> ENDCNT  =  1
--------> DMAADDR  =  0xa0040000
--------> DFXADDR  =  0xa0020000
--------> INTR_ENA =  0x1
--------> INTR     =  0x1
--------> ROUND_TRIP =  0x23d6
----- SLOT DATA ------------------
------> slot 0 :
        srcAddr   : 0x77dd0000,  srcSize   : 0xffc0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x474444
        loadMask  : 0b1
        storeMask : 0b110
        stIntrMask: 0b110
------> slot 1 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x77de0000,  desSize   : 0xffc0
        status    : 0x0
        profileCnt: 0x474442
        loadMask  : 0b110
        storeMask : 0b1
        stIntrMask: 0b1
------> slot 2 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
-----

In [24]:
buf_out.invalidate()
np_parRes = np.array(buf_out, dtype=np.int32)
print(np_parRes)

[[[[0.41015625]
   [0.4296875 ]
   [0.3955078 ]
   [0.5       ]]

  [[0.453125  ]
   [0.3955078 ]
   [0.453125  ]
   [0.5       ]]

  [[0.5       ]
   [0.48828125]
   [0.39941406]
   [0.5       ]]

  [[0.4765625 ]
   [0.4609375 ]
   [0.5       ]
   [0.484375  ]]]


 [[[0.47265625]
   [0.47265625]
   [0.4296875 ]
   [0.5       ]]

  [[0.44921875]
   [0.48828125]
   [0.46484375]
   [0.5       ]]

  [[0.48828125]
   [0.39941406]
   [0.44140625]
   [0.5       ]]

  [[0.5       ]
   [0.48046875]
   [0.42578125]
   [0.48046875]]]


 [[[0.40722656]
   [0.4296875 ]
   [0.4375    ]
   [0.4921875 ]]

  [[0.4609375 ]
   [0.4140625 ]
   [0.4609375 ]
   [0.5       ]]

  [[0.4765625 ]
   [0.36621094]
   [0.43359375]
   [0.5       ]]

  [[0.5       ]
   [0.48828125]
   [0.43359375]
   [0.46484375]]]


 ...


 [[[0.3876953 ]
   [0.44921875]
   [0.4033203 ]
   [0.4609375 ]]

  [[0.36621094]
   [0.4296875 ]
   [0.5       ]
   [0.5       ]]

  [[0.48046875]
   [0.39160156]
   [0.3203125 ]
   [0.5       ]

In [ ]:
print(f"Compare input_x and np_parRes: {np.array_equal(input_x, np_parRes)}")